# 06 - Feeding a model with PyTorch DataLoader

**Time:** about 30 minutes. **Before this:** Python loops and the process/backpressure lessons.

By the end, you can explain who loads a batch, avoid duplicate samples in a stream, and explain what prefetching and pinned memory do.

We generate eight tiny images. There are no downloads and no model to train. A **tensor** is a container for numbers, like a multidimensional array. You do not need a GPU. If PyTorch is absent, its examples print a skip message; the explanations and memory exercise still work.

Run the notebook from top to bottom. Keep the `lessons` folder beside it. Before each experiment, make a prediction.

**CUDA** is the software PyTorch uses to run work on NVIDIA GPUs. In this lesson, `cuda` names that kind of device; `cpu` names the processor used by normal Python code.

## Set up

PyTorch is optional for the rest of this course. To add it to this notebook's environment, use the command printed below, then restart the kernel and run again. Installation is a separate step; this notebook does not install packages automatically.

In [1]:
from pathlib import Path
import ast
import json
import os
import subprocess
import sys
from IPython.display import Code, display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "lessons" / "dataloader_demo.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the course folder, beside lessons/.")
helper = ROOT / "lessons" / "dataloader_demo.py"
helper_source = helper.read_text(encoding="utf-8")

try:
    import torch
except ModuleNotFoundError as error:
    if error.name != "torch":
        raise
    torch = None

TORCH_AVAILABLE = torch is not None
if TORCH_AVAILABLE:
    from torch.utils.data import DataLoader
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    from lessons.dataloader_demo import TinyImages
    print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
else:
    print("PyTorch examples will be skipped. Optional installation:")
    call_operator = '& ' if os.name == 'nt' else ''
    print(f'{call_operator}"{sys.executable}" -m pip install -r "{ROOT / "requirements-torch.txt"}"')

# Read the actual helper source so the worker code stays visible in the lesson.
def show_definition(name):
    tree = ast.parse(helper_source)
    node = next(item for item in tree.body if getattr(item, "name", None) == name)
    display(Code(ast.get_source_segment(helper_source, node), language="python"))

PyTorch: 2.9.1+cpu | CUDA available: False


## 1. A dataset gives samples; a loader groups them

A **sample** is one example, such as one image and its label. A **batch** is a group of samples used together. An **epoch** is one pass through the training data.

A map-style `Dataset` supports `dataset[index]`. Its `__len__` method reports the number of samples; `__getitem__` builds one sample. `DataLoader` collects samples into batches. Its default **collation** means stacking matching values together. Here, a dictionary of sample values becomes a dictionary of batched values. [PyTorch's dataset tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).

Read our dataset below. `worker=-1` is our label for the main process. `get_worker_info()` returns `None` there. The `pid` is the operating system's process ID.

An image **channel** holds one kind of value at each pixel. Grayscale uses one channel; RGB uses three for red, green, and blue. An image tensor has axes for channels, rows, and columns.

In [2]:
show_definition("TinyImages")

class TinyImages(Dataset):
    """A map-style dataset: request one sample using its index."""

    def __init__(self, count=8):
        self.count = count

    def __len__(self):
        return self.count

    def __getitem__(self, index):
        worker = get_worker_info()
        # A tiny generated image: 1 color channel, 2 rows, 2 columns.
        image = torch.full((1, 2, 2), float(index)) / max(self.count - 1, 1)
        return {
            "image": image,
            "index": index,
            "worker": -1 if worker is None else worker.id,
            "pid": os.getpid(),
        }

**Predict:** One image has shape `(1, 2, 2)`: one channel, two rows, two columns. What shape will two stacked images have? With `num_workers=0`, will any separate worker process load them?

In [3]:
if TORCH_AVAILABLE:
    dataset = TinyImages(count=8)
    loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=0)
    seen = []
    print("Notebook process ID:", os.getpid())
    for batch in loader:
        print("indices:", batch["index"].tolist(),
              "shape:", tuple(batch["image"].shape),
              "workers:", batch["worker"].tolist(),
              "process IDs:", batch["pid"].tolist())
        seen.extend(batch["index"].tolist())
        assert (batch["pid"] == os.getpid()).all().item()
    assert seen == list(range(8))
else:
    print("Skipped: install PyTorch to run the dataset example.")

Notebook process ID: 29044
indices: [0, 1] shape: (2, 1, 2, 2) workers: [-1, -1] process IDs: [29044, 29044]
indices: [2, 3] shape: (2, 1, 2, 2) workers: [-1, -1] process IDs: [29044, 29044]
indices: [4, 5] shape: (2, 1, 2, 2) workers: [-1, -1] process IDs: [29044, 29044]
indices: [6, 7] shape: (2, 1, 2, 2) workers: [-1, -1] process IDs: [29044, 29044]


**Interpret:** There are four batches. Each image batch has shape `(2, 1, 2, 2)`. The new first dimension counts the samples. Every process ID matches the notebook. This is a useful starting point when checking data or debugging an error.

## 2. Workers receive requests for indices

For a map-style dataset, the main process uses a **sampler** to choose indices and sends requests to workers. With `num_workers=2`, two separate processes load and transform samples. Workers receive dataset replicas, rather than permanently owning a fixed half of the indices. [PyTorch's multiprocessing data-loading behavior](https://docs.pytorch.org/docs/stable/data.html#multi-process-data-loading).

We request the same eight indices in forward and reverse order. Notice the `sampler=order` argument and the process IDs.

We run worker examples as a Python file. With **spawn**, each worker starts a fresh interpreter and imports the needed definitions. Classes live at the file's top level. The `if __name__ == "__main__"` guard prevents importing workers from launching more workers. This makes the example usable on Windows and from notebooks. [PyTorch's platform guidance](https://docs.pytorch.org/docs/stable/data.html#platform-specific-behaviors).

In [4]:
show_definition("run_map_demo")
show_definition("collect_rows")
print(helper_source[helper_source.rfind('if __name__ == "__main__":'):])

def run_map_demo():
    dataset = TinyImages(count=8)
    results = {}
    # The main process chooses these two index orders using the sampler.
    for label, order in (("forward", list(range(8))), ("reverse", list(reversed(range(8))))):
        loader = DataLoader(
            dataset,
            batch_size=2,
            sampler=order,
            num_workers=2,
            multiprocessing_context="spawn",
            prefetch_factor=1,
        )
        print(f"\nMap-style order: {label}; main process ID: {os.getpid()}")
        rows = collect_rows(loader)
        assert [row["index"] for row in rows] == order
        assert {row["worker"] for row in rows} == {0, 1}
        assert all(row["pid"] != os.getpid() for row in rows)
        results[label] = rows
    return results

def collect_rows(loader):
    """Read every batch so each worker can finish and shut down normally."""
    rows = []
    for batch in loader:
        print(
            "indices:", batch["index"].tolist(),
            "workers:", batch["worker"].tolist(),
            "process IDs:", batch["pid"].tolist(),
        )
        rows.extend(
            {"index": int(index), "worker": int(worker), "pid": int(pid)}
            for index, worker, pid in zip(batch["index"], batch["worker"], batch["pid"])
        )
    return rows

if __name__ == "__main__":
    mp.freeze_support()
    main()



The next function launches that file with this kernel's Python executable. Its final JSON line is text containing the results in a format Python can read. The launcher allows 180 seconds and shows the helper's output if it fails. A subprocess timeout stops this helper, but does not guarantee that every child process has also stopped.

In [5]:
def run_helper(mode):
    result = subprocess.run(
        [sys.executable, "-X", "utf8", str(helper), "--mode", mode],
        cwd=ROOT,
        text=True,
        capture_output=True,
        encoding="utf-8",
        timeout=180,
    )
    if result.returncode:
        raise RuntimeError(result.stdout + "\n" + result.stderr)
    lines = result.stdout.splitlines()
    for line in lines:
        if not line.startswith("RESULT_JSON="):
            print(line)
    if result.stderr:
        print(result.stderr)
    payload = next((line.removeprefix("RESULT_JSON=")
                    for line in lines if line.startswith("RESULT_JSON=")), None)
    if payload is None:
        raise RuntimeError("Helper returned no result:\n" + result.stdout + result.stderr)
    return json.loads(payload)

**Predict:** Will reversing the sampler change the order of returned samples? Must sample 0 always go to worker 0? Starting workers may take a few seconds; this tiny example is about behavior, not speed.

In [6]:
if TORCH_AVAILABLE:
    map_results = run_helper("map")["map"]
    for order, rows in map_results.items():
        print(order, "(index, worker):", [(row["index"], row["worker"]) for row in rows])
else:
    print("Skipped: install PyTorch to run the worker example.")


Map-style order: forward; main process ID: 30888
indices: [0, 1] workers: [0, 0] process IDs: [22604, 22604]
indices: [2, 3] workers: [1, 1] process IDs: [28300, 28300]
indices: [4, 5] workers: [0, 0] process IDs: [22604, 22604]
indices: [6, 7] workers: [1, 1] process IDs: [28300, 28300]

Map-style order: reverse; main process ID: 30888
indices: [7, 6] workers: [0, 0] process IDs: [20008, 20008]
indices: [5, 4] workers: [1, 1] process IDs: [18540, 18540]
indices: [3, 2] workers: [0, 0] process IDs: [20008, 20008]
indices: [1, 0] workers: [1, 1] process IDs: [18540, 18540]
forward (index, worker): [(0, 0), (1, 0), (2, 1), (3, 1), (4, 0), (5, 0), (6, 1), (7, 1)]
reverse (index, worker): [(7, 0), (6, 0), (5, 1), (4, 1), (3, 0), (2, 0), (1, 1), (0, 1)]


**Interpret:** The printed indices follow the requested order. The worker process IDs differ from the helper's main process ID. Compare the worker handling sample 0 across the two runs: in this example it changes when the order changes. Worker ID is a role within a loader, not permanent ownership of specific samples.

The code returns small tensors. Real image decoding would happen inside `__getitem__`, where the tiny image is currently made. More workers can help when this work is slow, but startup and communication also cost time.

## 3. Streams must divide their work

An `IterableDataset` supplies samples by yielding them from `__iter__`, like reading a stream of records. Each worker receives a replica. If both replicas yield 0 through 7, every number appears twice. **Sharding** means dividing the data into separate portions. `get_worker_info()` provides the worker ID and worker count for that division. [PyTorch's iterable dataset documentation](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.IterableDataset).

Read the code. With two workers, `range(start, count, step)` makes worker 0 yield even indices and worker 1 yield odd indices. With no worker, it yields every index.

In [7]:
show_definition("NumberStream")
show_definition("run_stream_demo")

class NumberStream(IterableDataset):
    """A stream, with an intentional duplicate-data option for teaching."""

    def __init__(self, count=8, shard=True):
        self.count = count
        self.shard = shard

    def __iter__(self):
        worker = get_worker_info()
        start, step = 0, 1
        if self.shard and worker is not None:
            # Worker 0 takes 0, 2, 4, 6; worker 1 takes 1, 3, 5, 7.
            start, step = worker.id, worker.num_workers
        for number in range(start, self.count, step):
            yield {
                "index": number,
                "worker": -1 if worker is None else worker.id,
                "pid": os.getpid(),
            }

def run_stream_demo():
    results = {}
    for shard in (False, True):
        loader = DataLoader(
            NumberStream(count=8, shard=shard),
            batch_size=2,
            num_workers=2,
            multiprocessing_context="spawn",
            prefetch_factor=1,
        )
        label = "sharded" if shard else "unsharded"
        print(f"\nIterable-style stream: {label}")
        rows = collect_rows(loader)
        counts = Counter(row["index"] for row in rows)
        expected_copies = 1 if shard else 2
        assert counts == Counter({index: expected_copies for index in range(8)})
        print("Occurrences per index:", dict(sorted(counts.items())))
        results[label] = rows
    return results

**Predict:** Without sharding, how many records will two workers return from this eight-record stream? After sharding, how many times will each index appear?

In [8]:
if TORCH_AVAILABLE:
    stream_results = run_helper("stream")["stream"]
    for name, rows in stream_results.items():
        print(name, "records:", len(rows),
              "unique indices:", len({row["index"] for row in rows}))
else:
    print("Skipped: install PyTorch to run the stream example.")


Iterable-style stream: unsharded
indices: [0, 1] workers: [0, 0] process IDs: [1464, 1464]
indices: [0, 1] workers: [1, 1] process IDs: [32088, 32088]
indices: [2, 3] workers: [0, 0] process IDs: [1464, 1464]
indices: [2, 3] workers: [1, 1] process IDs: [32088, 32088]
indices: [4, 5] workers: [0, 0] process IDs: [1464, 1464]
indices: [4, 5] workers: [1, 1] process IDs: [32088, 32088]
indices: [6, 7] workers: [0, 0] process IDs: [1464, 1464]
indices: [6, 7] workers: [1, 1] process IDs: [32088, 32088]
Occurrences per index: {0: 2, 1: 2, 2: 2, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2}

Iterable-style stream: sharded
indices: [0, 2] workers: [0, 0] process IDs: [19552, 19552]
indices: [1, 3] workers: [1, 1] process IDs: [31016, 31016]
indices: [4, 6] workers: [0, 0] process IDs: [19552, 19552]
indices: [5, 7] workers: [1, 1] process IDs: [31016, 31016]
Occurrences per index: {0: 1, 1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 1, 7: 1}
sharded records: 8 unique indices: 8
unsharded records: 16 unique indices: 8


**Interpret:** The unsharded version returns 16 records but only eight unique indices. The sharded version returns eight records, each once. It can return them in a different order from the map-style loader. Correct coverage matters more here than sorted order.

This stride rule works for our generated stream. With large files or remote sources, divide source files or partitions between workers when possible, so each worker does not read and discard most of the same input.

## 4. Pinned CPU memory and moving data to a GPU

`pin_memory=True` prepares tensors in **page-locked CPU memory**: memory the operating system keeps in RAM instead of moving to disk. The tensor is still on the CPU. Your loop moves it to the GPU with `.to(device, non_blocking=True)`.

`non_blocking=True` allows the CPU to continue without waiting for each transfer to finish. It does not guarantee simultaneous GPU copying and computation. Actual overlap needs suitable hardware, pinned input, and separate CUDA streams with correct coordination. A **stream** is an ordered queue of GPU operations. This beginner example uses the default stream. [PyTorch's pinned-memory and transfer guide](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html).

**Predict:** On a CUDA computer, will `batch["image"].device` already say `cuda` before `.to(...)`?

In [9]:
if TORCH_AVAILABLE:
    has_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if has_cuda else "cpu")
    transfer_loader = DataLoader(
        TinyImages(count=8),
        batch_size=2,
        num_workers=0,
        pin_memory=has_cuda,
    )
    for batch_number, batch in enumerate(transfer_loader):
        cpu_images = batch["image"]
        images = cpu_images.to(device, non_blocking=has_cuda)
        # A tiny computation on the chosen device; this is not model training.
        mean_value = images.mean()
        if batch_number == 0:
            print("Before transfer:", cpu_images.device, "| pinned:", cpu_images.is_pinned())
            print("After transfer:", images.device, "| shape:", tuple(images.shape))
        if has_cuda:
            # Finish the demo's pending work before reusing its inputs.
            torch.cuda.synchronize()
    print("Last batch mean:", mean_value.item())
    if not has_cuda:
        print("CPU fallback: GPU transfer and pinned-memory behavior were not exercised.")
else:
    print("Skipped: install PyTorch to run the transfer example.")

Before transfer: cpu | pinned: False
After transfer: cpu | shape: (2, 1, 2, 2)
Last batch mean: 0.9285714626312256
CPU fallback: GPU transfer and pinned-memory behavior were not exercised.


**Interpret:** CPU is the expected device before transfer, even with pinning. With no CUDA GPU, the cell stays on CPU and turns pinning off. That is enough to follow the data flow.

Keep worker outputs on CPU and move batches in the training process. Do not change pinned inputs while an asynchronous transfer is reading them. The explicit synchronization above keeps this demonstration simple; synchronizing every batch is not a recipe for maximum throughput.

## 5. Prefetching is a limited supply of work in advance

**Prefetching** means preparing future batches before the training loop asks for them. With automatic batching, `prefetch_factor` is batches **per worker**. Two workers with factor 2 can have about four batches in the prefetch pipeline. Pass this option only when `num_workers > 0`. [DataLoader's parameter documentation](https://docs.pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader).

The helper uses factor 1. Prefetch can hide uneven load times, but larger factors also keep more data in memory. Reducing workers, prefetch, or batch size can reduce that pressure. [PyTorch's data-loading optimization tutorial](https://docs.pytorch.org/tutorials/intermediate/intermediate_data_loading_tutorial.html).

**Predict:** If one batch contains 32 RGB images, each 224 by 224 with four-byte values, how much image storage do four batches need? This next cell does arithmetic only.

In [10]:
batch_size = 32
channels, height, width = 3, 224, 224
bytes_per_value = 4  # float32: a 32-bit floating-point number
workers = 2
prefetch_factor = 2

batch_bytes = batch_size * channels * height * width * bytes_per_value
prefetched_batches = workers * prefetch_factor
mib = 1024 ** 2
print(f"One batch's image values: {batch_bytes / mib:.2f} MiB")
print(f"Prefetch estimate: {prefetched_batches} batches")
print(f"Their image values alone: {prefetched_batches * batch_bytes / mib:.2f} MiB")

One batch's image values: 18.38 MiB
Prefetch estimate: 4 batches
Their image values alone: 73.50 MiB


**Interpret:** The estimate is 18.38 MiB per batch and 73.50 MiB for four batches. MiB means 1,048,576 bytes.

This is **not a bound on total RAM**. It excludes labels, Python objects, dataset replicas, temporary decoding buffers, batches held by your loop, and pinned copies. Model and GPU memory are separate again. Holding onto batches in a list can keep memory growing even with limited prefetch.

This connects to **backpressure**: when a consumer is slower than a producer, limit the amount waiting between them. DataLoader limits outstanding prefetched work rather than preparing the entire dataset in advance. If CPU workers already keep the GPU supplied, more workers cannot make that GPU compute faster. Measure the slow stage before adding more buffering.

## Practice

1. Change the first loader's batch size from 2 to 3. Predict the number of batches and the last batch's shape, then rerun that cell.
2. Imagine two stream workers both using `start=0, step=1`. Why is each sample repeated? Write the corrected values for each worker.
3. Each batch occupies 16 MiB. With two workers and `prefetch_factor=3`, estimate the prefetched batch storage. Change the factor to 1. Why are neither of these numbers a total RAM limit?
4. Your GPU is fully busy, while CPU workers spend time waiting. Would more workers help? What would you measure instead?
5. Explain the difference between `pin_memory=True` and `.to("cuda")` in one sentence.

Use the scratch cell for the arithmetic. Then open the answers.

In [11]:
# Try different values. This exercise does not require PyTorch.
exercise_batch_mib = 16
exercise_workers = 2
exercise_prefetch = 3
print("Estimated prefetched batch storage:",
      exercise_batch_mib * exercise_workers * exercise_prefetch, "MiB")

Estimated prefetched batch storage: 96 MiB


<details>
<summary>Worked answers</summary>

1. Eight samples form batches of 3, 3, and 2. There are three batches. The final image tensor has shape `(2, 1, 2, 2)`, because the default keeps the shorter final batch.
2. Both replicas walk over all eight records. Worker 0 needs `start=0, step=2`; worker 1 needs `start=1, step=2`. Their index sets then do not overlap.
3. `16 * 2 * 3 = 96 MiB`; with factor 1, `16 * 2 * 1 = 32 MiB`. Dataset objects, current batches, decoding buffers, and pinned copies still need memory.
4. Extra workers are unlikely to help a GPU that already has work. Measure batch wait time, overall examples per second, and memory use. Try fewer workers or less prefetch and check whether throughput stays the same. These tiny demonstrations cannot choose settings for a real training job.
5. Pinning prepares CPU memory for transfer; `.to("cuda")` creates the tensor on the GPU.

</details>

## Explain it back

You have eight CPU cores and many images. Explain this plan aloud:

- Start with `num_workers=0` to check that the data is correct.
- Try a small number of workers for decoding and transforms, then measure. CPU core count alone does not determine the best worker count.
- Let the main sampler choose map-style indices. Explicitly divide iterable streams.
- Use small, controlled prefetch. Account for memory beyond the queued image tensors.
- On CUDA, consider pinned CPU batches and move them in the training process.

An **input bottleneck** means the model is waiting for data. A **compute bottleneck** means the model's own computation sets the pace. Choosing between them is a measurement question. More concurrency helps only when it addresses the slow part of the pipeline.

Continue with [07 · Measuring performance](07_measuring_performance.ipynb) for the advanced lessons, or return to the [course guide](README.md).